# Fine-tune East Frisian (Oostfräisk) TTS with Piper

This notebook fine-tunes a VITS model from [Piper](https://github.com/rhasspy/piper) on East Frisian Low Saxon data.

**Two training modes available:**

| Mode | Phonemizer | Pros | Cons |
|------|-----------|------|------|
| **`grapheme`** | None — raw characters | Simple, no preprocessing hack, keeps original spelling | Needs more data to learn pronunciation patterns |
| **`espeak`** | espeak-ng (German) | Leverages German pronunciation knowledge | Requires text preprocessing (ğ→ch, ó→oa, etc.) |

Set your preferred mode in the cell below.

**Requirements:** Google Colab with A100 GPU

In [28]:
# =============================================
# CONFIGURATION — set your preferred mode here
# =============================================

PHONEME_MODE = "grapheme"   # "grapheme" or "espeak"

# Training hyperparameters
BATCH_SIZE = 16
MAX_EPOCHS = 3000            # safety net — early stopping will kick in sooner
QUALITY = "medium"          # "medium" (22050 Hz) or "high" (22050 Hz, larger model)
EARLY_STOPPING_PATIENCE = 50  # stop after 50 epochs with no val_loss improvement

print(f"Mode: {PHONEME_MODE}")
print(f"Batch size: {BATCH_SIZE}, Max epochs: {MAX_EPOCHS}, Quality: {QUALITY}")
print(f"Early stopping patience: {EARLY_STOPPING_PATIENCE} epochs")

Mode: grapheme
Batch size: 16, Max epochs: 3000, Quality: medium
Early stopping patience: 50 epochs


# 1. Install Dependencies

In [2]:
# System packages needed for building piper
!apt-get install -y espeak-ng build-essential cmake ninja-build

# Install Piper from the new OHF-Voice repo (includes training code)
!pip install -U pip
!git clone --depth 1 https://github.com/OHF-Voice/piper1-gpl.git /tmp/piper
!cd /tmp/piper && pip install -e '.[train]'
!cd /tmp/piper && bash build_monotonic_align.sh
!cd /tmp/piper && python setup.py build_ext --inplace

# Install piper-tts for inference testing
!pip install piper-tts

# HuggingFace Hub for checkpoint download
!pip install huggingface_hub

E: Could not open lock file /var/lib/dpkg/lock-frontend - open (13: Permission denied)
E: Unable to acquire the dpkg frontend lock (/var/lib/dpkg/lock-frontend), are you root?
fatal: destination path '/tmp/piper' already exists and is not an empty directory.
Obtaining file:///tmp/piper
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for piper-tts (pyproject.toml) ... done
  Created wheel for piper-tts: filename=piper_tts-1.4.2-0.editable-py3-none-any.whl size=14459 sha256=7cfbfb080dddaa208b2ea9ba72534a102504e12f8fe542e9975593bc911354e7
  Stored in directory: /tmp/pip-ephem-wheel-cache-8vysgs7z/wheels/2b/6f/ab/dadddad1b5ff4b8d929cf9fbb52bc30dead7ffa2cf64c4e1e4
Successfully built piper-tts
  Attempting uninstall: piper-tts
    Found existing installation: piper-tts 1.4.2
    Uninstalling piper-tts-1.4.2:
      

In [29]:
# ── PATCH piper1-gpl bugs (idempotent) ───────────────────────────────

dataset_py = "/tmp/piper/src/piper/train/vits/dataset.py"
export_py = "/tmp/piper/src/piper/train/export_onnx.py"

# Bug 1: Custom phoneme map (--data.phonemes_path) is loaded but never
# passed to phonemes_to_ids(), so training always uses DEFAULT_PHONEME_ID_MAP.
# This silently drops uppercase letters + combining diacritical marks (̈ ̂ ̆ ́).
with open(dataset_py, "r") as f:
    lines = f.readlines()

# Use a unique marker to check if patch was already applied
PATCH_MARKER = "# PATCH-PHONEME-MAP: Use custom phoneme map for phoneme-to-ID conversion"

if not any(PATCH_MARKER in line for line in lines):
    # Find the "elif self.phoneme_type == PhonemeType.PINYIN:" line
    # and insert our patch BEFORE it (inside the 'if self.phonemes_path:' block)
    insert_idx = None
    for i, line in enumerate(lines):
        if "elif self.phoneme_type == PhonemeType.PINYIN:" in line:
            insert_idx = i
            break

    if insert_idx is not None:
        patch_lines = [
            "\n",
            "            " + PATCH_MARKER + "\n",
            "            _custom_map = phoneme_id_map\n",
            "            phonemes_to_ids = lambda phonemes: default_phonemes_to_ids(phonemes, id_map=_custom_map)\n",
            "\n",
        ]
        for j, pl in enumerate(patch_lines):
            lines.insert(insert_idx + j, pl)

        with open(dataset_py, "w") as f:
            f.writelines(lines)
        print("✅ Patched dataset.py: phonemes_to_ids() now uses custom phoneme map")
    else:
        print("❌ ERROR: Could not find insertion point in dataset.py!")
else:
    print("✅ dataset.py already patched")

# Bug 2: PyTorch 2.6+ defaults to dynamo-based ONNX exporter which can't
# trace VITS's data-dependent asserts. Force legacy TorchScript exporter.
with open(export_py, "r") as f:
    src = f.read()

if "dynamo=False" not in src:
    src = src.replace("verbose=False,", "dynamo=False,\n        verbose=False,")
    with open(export_py, "w") as f:
        f.write(src)
    print("✅ Patched export_onnx.py: using legacy TorchScript ONNX exporter")
else:
    print("✅ export_onnx.py already patched")

✅ dataset.py already patched
✅ export_onnx.py already patched


# 2. Clone Repository & Setup

In [ ]:
!git clone https://VanModers:@github.com/VanModers/oostfraeisk_text_to_speech
%cd oostfraeisk_text_to_speech
!git pull

# 3. GPU Optimizations

In [30]:
import torch

torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True
torch.backends.cudnn.benchmark = True

print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
print(f"TF32 enabled: {torch.backends.cuda.matmul.allow_tf32}")

GPU: NVIDIA RTX PRO 6000 Blackwell Workstation Edition
VRAM: 102.0 GB
TF32 enabled: True


# 4. Download Pretrained German Checkpoint

Fine-tune from the German **Thorsten** voice (medium quality = VITS @ 22050 Hz).

In [31]:
from huggingface_hub import hf_hub_download, list_repo_tree
import os

repo_id = "rhasspy/piper-checkpoints"

# Find the German Thorsten medium checkpoint
all_files = [
    f.rfilename for f in list_repo_tree(repo_id, repo_type="dataset")
    if hasattr(f, 'rfilename')
    and "thorsten" in f.rfilename
    and "medium" in f.rfilename
    and f.rfilename.endswith(".ckpt")
]
print("Available Thorsten medium checkpoints:")
for f in all_files:
    print(f"  {f}")

if all_files:
    pretrained_ckpt = hf_hub_download(
        repo_id=repo_id, filename=all_files[0], repo_type="dataset"
    )
else:
    pretrained_ckpt = hf_hub_download(
        repo_id=repo_id,
        filename="de/de_DE/thorsten/medium/epoch=3135-step=2702056.ckpt",
        repo_type="dataset"
    )

print(f"\nDownloaded: {pretrained_ckpt}")

Available Thorsten medium checkpoints:

Downloaded: /home/tidospecht/.cache/huggingface/hub/datasets--rhasspy--piper-checkpoints/snapshots/52588227e5a29f8c2afc6c31280e42119760ac86/de/de_DE/thorsten/medium/epoch=3135-step=2702056.ckpt


# 5. Preprocess Text (espeak mode only)

In the new Piper, preprocessing (phonemization + audio caching) is handled automatically during training. No separate preprocessing step is needed.

- **`grapheme`**: No text preprocessing. The trainer uses `--data.phoneme_type text` to treat raw characters as phonemes.
- **`espeak`**: We still need to convert East Frisian orthography → German-compatible forms in the CSV before training, since the trainer will use espeak-ng with a German voice.

In [32]:
# East Frisian → German phoneme mapping (only used in espeak mode)
custom_phoneme_map = {
    # Complex diphthongs / triphthongs (longest first)
    "öye": "öije",    # /œyə/ - göyen (gießen)
    "ööe": "ööö",    # extra-long ö
    "óóej": "ooai",  # /ɒ:ɛɪ/ - dóóejt (Tat)
    "âau": "aau",    # /a:ʊ/ - brâau
    "âaj": "aai",    # /a:ɪ/ - drâajen (drehen)
    "êer": "eer",    # /e:r/ - fêert (fährt)
    "êel": "eel",    # /e:l/ - fêelen (fühlen)

    # Circumflex (extra-long) vowels
    "ââ": "aa", "êê": "ee", "îî": "ii", "ôô": "oo", "ûû": "uu",
    "âa": "aa", "êe": "ee", "îi": "ii", "ôo": "oo", "ûu": "uu",

    # East Frisian specific vowels
    "óój": "oai", "óó": "oa", "ó": "oa",

    # ö-diphthongs
    "öy": "öi", "öej": "ööi", "öj": "öi",

    # ä-diphthongs
    "äie": "ääi", "äej": "ääi", "äj": "äi", "äi": "äi",

    # Basic diphthongs
    "ooj": "ooi", "oi": "oi", "ei": "ei",
    "aaj": "aai", "ai": "ai", "aau": "aau", "au": "au",

    # Consonants
    "ğ": "ch", "tj": "tsch",
}

_sorted_keys = sorted(custom_phoneme_map.keys(), key=len, reverse=True)

def preprocess_east_frisian(text: str) -> str:
    """Convert East Frisian orthography to German-compatible forms."""
    result = text
    for key in _sorted_keys:
        result = result.replace(key, custom_phoneme_map[key])
    return result

# Quick test
for t in ["Suldóót", "fandóóeğ", "drâajen"]:
    print(f"  {t} → {preprocess_east_frisian(t)}")

  Suldóót → Suldoat
  fandóóeğ → fandoaech
  drâajen → draaien


In [33]:
import shutil
from pathlib import Path

DATASET_DIR = Path("data/oostfraeisk")
metadata_path = DATASET_DIR / "metadata.csv"

if PHONEME_MODE == "espeak":
    # ── ESPEAK MODE ─────────────────────────────────────
    # Preprocess text: East Frisian → German-compatible
    backup_path = metadata_path.with_suffix('.csv.original')
    if not backup_path.exists():
        shutil.copy(metadata_path, backup_path)
        print(f"Backed up original to {backup_path}")
    else:
        shutil.copy(backup_path, metadata_path)
        print(f"Restored original from {backup_path}")

    with open(metadata_path, 'r', encoding='utf-8') as f:
        lines = f.readlines()

    processed_lines = []
    for line in lines:
        parts = line.strip().split('|')
        if len(parts) >= 2:
            filename = parts[0]
            text = parts[1]
            processed = preprocess_east_frisian(text)
            if len(parts) == 3:
                processed_lines.append(f"{filename}|{processed}|{processed}\n")
            else:
                processed_lines.append(f"{filename}|{processed}\n")
        else:
            processed_lines.append(line)

    with open(metadata_path, 'w', encoding='utf-8') as f:
        f.writelines(processed_lines)

    print(f"Preprocessed {len(processed_lines)} lines for espeak mode")
    print(f"Example: {lines[0].strip()}")
    print(f"       → {processed_lines[0].strip()}")

elif PHONEME_MODE == "grapheme":
    print("Grapheme mode — no text preprocessing needed.")
    print("Original East Frisian text will be used as-is.")
    print("The trainer will handle character-to-ID mapping automatically.")
else:
    raise ValueError(f"Unknown PHONEME_MODE: {PHONEME_MODE}")

Grapheme mode — no text preprocessing needed.
Original East Frisian text will be used as-is.
The trainer will handle character-to-ID mapping automatically.


In [34]:
# Verify dataset is ready
import csv
from pathlib import Path

DATASET_DIR = Path("data/oostfraeisk")
wav_dir = DATASET_DIR / "wavs"
if not wav_dir.is_dir():
    wav_dir = DATASET_DIR / "wav"

metadata_path = DATASET_DIR / "metadata.csv"
num_lines = 0
missing = 0
with open(metadata_path, 'r', encoding='utf-8') as f:
    reader = csv.reader(f, delimiter='|')
    for row in reader:
        num_lines += 1
        filename = row[0]
        wav_path = wav_dir / f"{filename}.wav"
        if not wav_path.exists():
            wav_path = wav_dir / filename
        if not wav_path.exists():
            missing += 1
            if missing <= 5:
                print(f"WARNING: Missing {filename}")

print(f"\nDataset: {num_lines} utterances, {missing} missing audio files")
print(f"Audio dir: {wav_dir}")
print(f"Sample: ", end="")
with open(metadata_path, 'r', encoding='utf-8') as f:
    print(f.readline().strip())


Dataset: 1004 utterances, 0 missing audio files
Audio dir: data/oostfraeisk/wavs
Sample: sentence_0001|Ennerwor mank däi fräej mäiden tüsken 't Braukmer- un 't Auerkerland wor man dat lûud|Ennerwor mank däi fräej mäiden tüsken 't Braukmer- un 't Auerkerland wor man dat lûud


In [35]:
import csv
import json
import unicodedata
from pathlib import Path
from collections import Counter

DATASET_DIR = Path("data/oostfraeisk")
TRAINING_DIR = Path("piper_training")
TRAINING_DIR.mkdir(parents=True, exist_ok=True)

metadata_path = DATASET_DIR / "metadata.csv"

if PHONEME_MODE == "grapheme":
    # ── Build custom phoneme ID map for grapheme mode ───────
    # Piper does unicodedata.normalize("NFD", text) which decomposes
    # accented chars: ä→a+̈, ó→o+́, â→a+̂, ğ→g+̆
    # The default map only has basic lowercase + IPA.
    # We need to add: uppercase letters, combining diacritical marks,
    # and any other characters in our text.

    all_chars = Counter()
    with open(metadata_path, 'r', encoding='utf-8') as f:
        reader = csv.reader(f, delimiter='|')
        for row in reader:
            text = row[-1]
            # This is exactly what Piper does internally
            nfd_text = unicodedata.normalize("NFD", text)
            all_chars.update(nfd_text)

    print(f"Unique characters after NFD normalization ({len(all_chars)}):")
    for char, count in sorted(all_chars.items(), key=lambda x: -x[1]):
        name = unicodedata.name(char, f"U+{ord(char):04X}")
        print(f"  {repr(char):10s} (U+{ord(char):04X} {name}): {count}x")

    # Start from Piper's default map (IPA + basic Latin)
    from piper.phoneme_ids import DEFAULT_PHONEME_ID_MAP

    phoneme_id_map = dict(DEFAULT_PHONEME_ID_MAP)
    next_id = max(max(ids) for ids in phoneme_id_map.values()) + 1

    # Add any missing characters from our dataset
    missing = []
    for char in sorted(all_chars.keys()):
        if char not in phoneme_id_map:
            phoneme_id_map[char] = [next_id]
            missing.append((char, next_id))
            next_id += 1

    if missing:
        print(f"\nAdded {len(missing)} characters to phoneme ID map:")
        for char, pid in missing:
            name = unicodedata.name(char, f"U+{ord(char):04X}")
            print(f"  {repr(char):10s} → ID {pid}  ({name})")
    else:
        print("\nNo missing characters — default map covers everything.")

    # Save the custom map
    phonemes_path = TRAINING_DIR / "phoneme_id_map.json"
    with open(phonemes_path, "w", encoding="utf-8") as f:
        json.dump(phoneme_id_map, f, ensure_ascii=False, indent=2)

    num_symbols = max(max(ids) for ids in phoneme_id_map.values()) + 1
    print(f"\nSaved phoneme ID map to {phonemes_path}")
    print(f"Total entries: {len(phoneme_id_map)}, num_symbols needed: {num_symbols}")

else:
    print("espeak mode — using Piper's built-in phoneme ID map.")
    phonemes_path = None
    num_symbols = 256

Unique characters after NFD normalization (79):
  ' '        (U+0020 SPACE): 15225x
  'e'        (U+0065 LATIN SMALL LETTER E): 7669x
  'n'        (U+006E LATIN SMALL LETTER N): 7368x
  'a'        (U+0061 LATIN SMALL LETTER A): 6409x
  'o'        (U+006F LATIN SMALL LETTER O): 6298x
  'i'        (U+0069 LATIN SMALL LETTER I): 5691x
  't'        (U+0074 LATIN SMALL LETTER T): 5332x
  'r'        (U+0072 LATIN SMALL LETTER R): 4186x
  '̈'        (U+0308 COMBINING DIAERESIS): 4074x
  's'        (U+0073 LATIN SMALL LETTER S): 3884x
  'u'        (U+0075 LATIN SMALL LETTER U): 3741x
  'd'        (U+0064 LATIN SMALL LETTER D): 3634x
  'l'        (U+006C LATIN SMALL LETTER L): 2629x
  'k'        (U+006B LATIN SMALL LETTER K): 2529x
  'm'        (U+006D LATIN SMALL LETTER M): 2078x
  '́'        (U+0301 COMBINING ACUTE ACCENT): 1928x
  "'"        (U+0027 APOSTROPHE): 1705x
  'f'        (U+0066 LATIN SMALL LETTER F): 1596x
  'g'        (U+0067 LATIN SMALL LETTER G): 1560x
  'h'        (U+0068 LATI

# 6. Train the Model

Fine-tune from the pretrained German Thorsten checkpoint using the new `piper.train fit` CLI.

**Important:** Since the German checkpoint uses espeak phonemes and our grapheme model uses raw characters (different phoneme set), we use `--model.vocoder_warmstart_ckpt` for grapheme mode — this only copies the vocoder/acoustic parameters, not the phoneme embedding layer.

For espeak mode (where we've converted text to German-compatible forms), we can use `--ckpt_path` to load the full checkpoint.

**Tips:**
- **Early stopping** monitors `val_loss` and stops when it plateaus (patience = 50 epochs)
- **ModelCheckpoint** saves top 3 checkpoints by `val_loss` — no more losing the best model
- Adjust `--data.batch_size` if you run out of VRAM
- On ~1000 utterances, expect the model to peak around epoch 100-200

In [36]:
from pathlib import Path

DATASET_DIR = Path("data/oostfraeisk")
TRAINING_DIR = Path("piper_training")
TRAINING_DIR.mkdir(parents=True, exist_ok=True)

config_path = TRAINING_DIR / "config.json"
cache_dir = TRAINING_DIR / "cache"

# Build the training command
base_cmd = [
    "python -m piper.train fit",
    f'--data.voice_name "oostfraeisk"',
    f"--data.csv_path {DATASET_DIR / 'metadata.csv'}",
    f"--data.audio_dir {DATASET_DIR / 'wavs'}",
    f"--model.sample_rate 22050",
    f"--data.espeak_voice de",
    f"--data.cache_dir {cache_dir}",
    f"--data.config_path {config_path}",
    f"--data.batch_size {BATCH_SIZE}",
    f"--data.validation_split 0.05",
    f"--data.num_test_examples 5",
    f"--trainer.max_epochs {MAX_EPOCHS}",
    "--trainer.accelerator gpu",
    "--trainer.devices 1",
    "--trainer.precision 32",
    # Save top 3 checkpoints by val_loss (best model won't be lost)
    #"--trainer.callbacks+=ModelCheckpoint",
    #"--trainer.callbacks.monitor=val_loss",
    #"--trainer.callbacks.mode=min",
    #"--trainer.callbacks.save_top_k=3",
    #"--trainer.callbacks.every_n_epochs=5",
    # Early stopping — stop when val_loss stops improving
    #"--trainer.callbacks+=EarlyStopping",
    #"--trainer.callbacks.monitor=val_loss",
    #f"--trainer.callbacks.patience={EARLY_STOPPING_PATIENCE}",
]

if PHONEME_MODE == "grapheme":
    base_cmd.append("--data.phoneme_type text")
    # Custom phoneme map with East Frisian characters + combining marks
    base_cmd.append(f"--data.phonemes_path {TRAINING_DIR / 'phoneme_id_map.json'}")
    base_cmd.append(f"--data.num_symbols {num_symbols}")
    # Use vocoder warmstart — different phoneme set than German checkpoint
    base_cmd.append(f'--model.vocoder_warmstart_ckpt "{pretrained_ckpt}"')
else:
    # Use full checkpoint — same espeak German phonemes
    base_cmd.append(f'--ckpt_path "{pretrained_ckpt}"')

cmd = " \\\n    ".join(base_cmd)
print("Training command:")
print(cmd)

Training command:
python -m piper.train fit \
    --data.voice_name "oostfraeisk" \
    --data.csv_path data/oostfraeisk/metadata.csv \
    --data.audio_dir data/oostfraeisk/wavs \
    --model.sample_rate 22050 \
    --data.espeak_voice de \
    --data.cache_dir piper_training/cache \
    --data.config_path piper_training/config.json \
    --data.batch_size 16 \
    --data.validation_split 0.05 \
    --data.num_test_examples 5 \
    --trainer.max_epochs 3000 \
    --trainer.accelerator gpu \
    --trainer.devices 1 \
    --trainer.precision 32 \
    --data.phoneme_type text \
    --data.phonemes_path piper_training/phoneme_id_map.json \
    --data.num_symbols 196 \
    --model.vocoder_warmstart_ckpt "/home/tidospecht/.cache/huggingface/hub/datasets--rhasspy--piper-checkpoints/snapshots/52588227e5a29f8c2afc6c31280e42119760ac86/de/de_DE/thorsten/medium/epoch=3135-step=2702056.ckpt"


In [37]:
# Clear cached phoneme IDs to ensure fresh conversion with patched code
!rm -rf piper_training/cache

!python -m piper.train fit \
    --data.voice_name "oostfraeisk" \
    --data.csv_path data/oostfraeisk/metadata.csv \
    --data.audio_dir data/oostfraeisk/wavs \
    --model.sample_rate 22050 \
    --data.espeak_voice de \
    --data.cache_dir piper_training/cache \
    --data.config_path piper_training/config.json \
    --data.batch_size 16 \
    --data.validation_split 0.05 \
    --data.num_test_examples 5 \
    --trainer.max_epochs 3000 \
    --trainer.accelerator gpu \
    --trainer.devices 1 \
    --trainer.precision 32 \
    --data.phoneme_type text \
    --data.phonemes_path piper_training/phoneme_id_map.json \
    --data.num_symbols 196 \
    --model.vocoder_warmstart_ckpt "/home/tidospecht/.cache/huggingface/hub/datasets--rhasspy--piper-checkpoints/snapshots/52588227e5a29f8c2afc6c31280e42119760ac86/de/de_DE/thorsten/medium/epoch=3135-step=2702056.ckpt"

/home/tidospecht/repos/oostfraeisk_text_to_speech/.venv/lib/python3.10/site-packages/lightning/fabric/utilities/seed.py:44: No seed found, seed set to 0
Seed set to 0
/home/tidospecht/repos/oostfraeisk_text_to_speech/.venv/lib/python3.10/site-packages/torch/nn/utils/weight_norm.py:144: FutureWarning: `torch.nn.utils.weight_norm` is deprecated in favor of `torch.nn.utils.parametrizations.weight_norm`.
  WeightNorm.apply(module, name, dim)
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
INFO:piper.train.vits.dataset:Processing ut

In [ ]:
import subprocess, shlex

# Run the training command
result = subprocess.run(cmd, shell=True, check=False)
if result.returncode != 0:
    print(f"\nTraining exited with code {result.returncode}")
else:
    print("\nTraining complete!")

# 7. Find Best Checkpoint

In [38]:
import glob
import re

# Check both possible checkpoint locations
ckpts = sorted(
    glob.glob("lightning_logs/version_*/checkpoints/*.ckpt") +
    glob.glob(str(TRAINING_DIR / "lightning_logs/version_*/checkpoints/*.ckpt"))
)

print("Available checkpoints:")
for ckpt in ckpts[-10:]:
    print(f"  {ckpt}")

if ckpts:
    # Pick the checkpoint with the lowest val_loss (filename includes val_loss=XX.XX)
    best_ckpt = None
    best_val_loss = float("inf")
    for ckpt in ckpts:
        m = re.search(r"val_loss=([\d.]+)", ckpt)
        if m:
            val_loss = float(m.group(1))
            if val_loss < best_val_loss:
                best_val_loss = val_loss
                best_ckpt = ckpt

    if best_ckpt:
        print(f"\nUsing best checkpoint (val_loss={best_val_loss:.2f}): {best_ckpt}")
    else:
        # Fallback: use last checkpoint if filenames don't contain val_loss
        best_ckpt = ckpts[-1]
        print(f"\nNo val_loss in filenames, using latest: {best_ckpt}")
else:
    print("No checkpoints found!")

Available checkpoints:
  lightning_logs/version_0/checkpoints/epoch=2999-step=360000.ckpt

No val_loss in filenames, using latest: lightning_logs/version_0/checkpoints/epoch=2999-step=360000.ckpt


# 8. Export to ONNX

Creates `model.onnx` for fast CPU inference using the new `piper.train.export_onnx` module. The config JSON written during training is copied alongside as `model.onnx.json`.

In [16]:
!pip install onnxscript

In [39]:
import os

!mkdir -p model_piper

!python -m piper.train.export_onnx \
    --checkpoint "{best_ckpt}" \
    --output-file model_piper/oostfraeisk.onnx

# Copy training config as ONNX config
!cp {str(TRAINING_DIR)}/config.json model_piper/oostfraeisk.onnx.json

onnx_path = "model_piper/oostfraeisk.onnx"
if os.path.exists(onnx_path):
    size_mb = os.path.getsize(onnx_path) / 1e6
    print(f"\nExported: {onnx_path} ({size_mb:.1f} MB)")
else:
    print("Export failed!")

/home/tidospecht/repos/oostfraeisk_text_to_speech/.venv/lib/python3.10/site-packages/torch/nn/utils/weight_norm.py:144: FutureWarning: `torch.nn.utils.weight_norm` is deprecated in favor of `torch.nn.utils.parametrizations.weight_norm`.
  WeightNorm.apply(module, name, dim)
/tmp/piper/src/piper/train/export_onnx.py:92: DeprecationWarning: You are using the legacy TorchScript-based ONNX export. Starting in PyTorch 2.9, the new torch.export-based ONNX exporter has become the default. Learn more about the new export logic: https://docs.pytorch.org/docs/stable/onnx_export.html. For exporting control flow: https://pytorch.org/tutorials/beginner/onnx/export_control_flow_model_to_onnx_tutorial.html
  torch.onnx.export(
/tmp/piper/src/piper/train/vits/attentions.py:235: TracerWarning: Converting a tensor to a Python boolean might cause the trace to be incorrect. We can't record the data flow of Python values, so this value will be treated as a constant in the future. This means that the trace 

# 9. Test Inference

In [40]:
from piper import PiperVoice
import wave

voice = PiperVoice.load("model_piper/oostfraeisk.onnx")
print(f"Model loaded! Sample rate: {voice.config.sample_rate}")
print(f"Phoneme type: {voice.config.phoneme_type}")

Model loaded! Sample rate: 22050
Phoneme type: text


In [41]:
test_sentences = [
    "Moin, woo gaajt 't dii?",
    "Denkent jii, dat ik disser sats gaud uutprooten dau?",
    "Hest duu däi süen fandóóeğ al säin?",
    "Däi oorsprungelk tóól fan däi Fräisen tüsken Laauwers un Wäiser was dat Olfräisk.",
    "In disser sats gift dat kiin umluuden of anner roer dingen, man disser is trotsdeem langer as gewoon um dat tau testen.",
]

for i, text in enumerate(test_sentences):
    # In espeak mode, preprocess; in grapheme mode, use as-is
    synth_text = preprocess_east_frisian(text) if PHONEME_MODE == "espeak" else text
    out_path = f"test_piper_{i}.wav"
    with wave.open(out_path, "w") as wav_file:
        # piper1-gpl training package shadows piper-tts and has a different API:
        # synthesize() returns a generator of AudioChunks, synthesize_wav() writes to file
        if hasattr(voice, 'synthesize_wav'):
            voice.synthesize_wav(synth_text, wav_file)
        else:
            voice.synthesize(synth_text, wav_file)
    print(f"[{i}] {text}")
    if PHONEME_MODE == "espeak":
        print(f"    → {synth_text}")
    print()

[0] Moin, woo gaajt 't dii?

[1] Denkent jii, dat ik disser sats gaud uutprooten dau?

[2] Hest duu däi süen fandóóeğ al säin?

[3] Däi oorsprungelk tóól fan däi Fräisen tüsken Laauwers un Wäiser was dat Olfräisk.

[4] In disser sats gift dat kiin umluuden of anner roer dingen, man disser is trotsdeem langer as gewoon um dat tau testen.



In [42]:
import IPython
IPython.display.Audio("test_piper_0.wav")

In [43]:
IPython.display.Audio("test_piper_1.wav")

In [44]:
IPython.display.Audio("test_piper_2.wav")

In [45]:
IPython.display.Audio("test_piper_3.wav")


# Part 2: Multi-speaker Training

The recordings come from two different speakers across non-contiguous ranges:

| Range | Speaker |
|-------|---------|
| 1 – 300 | speaker_1 |
| 301 – 400 | speaker_2 |
| 401 – 500 | speaker_1 |
| 501 – 600 | speaker_2 |
| 601 – 650 | speaker_1 |
| 651 – 850 | speaker_2 |
| 851 – 864 | speaker_1 |
| 865 – 1004 | speaker_2 |

**Strategy:** Train a 2-speaker VITS model with explicit speaker IDs. After training, average the two learned speaker embeddings and export a **single-speaker ONNX** — the averaged embedding blends both voice characteristics instead of randomly collapsing to one.

**Prerequisites:** Run sections 1–5 first (install deps, clone repo, GPU setup, checkpoint download, phoneme map).

## Part 2a — Generate Multi-speaker Metadata

Creates `data/oostfraeisk/metadata_multispeaker.csv` with format `sentence_XXXX|speaker_N|text`.


In [ ]:

import csv
import re
from pathlib import Path

DATASET_DIR = Path("data/oostfraeisk")

# Speaker assignment: (first_sentence_num, last_sentence_num, speaker_name)
SPEAKER_RANGES = [
    (1,    300,  "speaker_1"),
    (301,  400,  "speaker_2"),
    (401,  500,  "speaker_1"),
    (501,  600,  "speaker_2"),
    (601,  650,  "speaker_1"),
    (651,  850,  "speaker_2"),
    (851,  864,  "speaker_1"),
    (865,  9999, "speaker_2"),
]

def get_speaker(n):
    for start, end, speaker in SPEAKER_RANGES:
        if start <= n <= end:
            return speaker
    return "speaker_1"

metadata_path       = DATASET_DIR / "metadata.csv"
multi_metadata_path = DATASET_DIR / "metadata_multispeaker.csv"

# Format: sentence_XXXX|speaker_N|text
# piper1-gpl reads speaker from row[1] and text from row[-1]
with open(metadata_path, "r", encoding="utf-8") as f_in, \
     open(multi_metadata_path, "w", encoding="utf-8") as f_out:
    for row in csv.reader(f_in, delimiter="|"):
        utt_id  = row[0]
        text    = row[-1]
        m       = re.match(r"sentence_(\d+)", utt_id)
        speaker = get_speaker(int(m.group(1))) if m else "speaker_1"
        f_out.write(f"{utt_id}|{speaker}|{text}\n")

print(f"Generated {multi_metadata_path}")

# Spot-check speaker boundaries
print("\nSpot-check around speaker transitions:")
with open(multi_metadata_path) as f:
    lines = f.readlines()
for i in [298, 299, 300, 301, 399, 400, 401, 499, 500, 501]:
    if i < len(lines):
        print(f"  [{i+1:4d}] {lines[i].rstrip()[:80]}")



## Part 2b — Train Multi-speaker Model

Uses the same phoneme map built in section 5.
Checkpoints are saved to `piper_training_multi_logs/` (separate from single-speaker).

**Note:** `--model.num_speakers 2` enables the speaker embedding layer.
The vocoder warmstart copies acoustic/vocoder weights from the German checkpoint
but initialises the speaker embeddings from scratch.


In [ ]:

# Clear cache so phoneme IDs are rebuilt with the patched code
!rm -rf piper_training_multi/cache

!python -m piper.train fit \
    --data.voice_name "oostfraeisk" \
    --data.csv_path data/oostfraeisk/metadata_multispeaker.csv \
    --data.audio_dir data/oostfraeisk/wavs \
    --model.sample_rate 22050 \
    --data.espeak_voice de \
    --data.cache_dir piper_training_multi/cache \
    --data.config_path piper_training_multi/config.json \
    --data.batch_size {BATCH_SIZE} \
    --data.validation_split 0.05 \
    --data.num_test_examples 5 \
    --trainer.max_epochs {MAX_EPOCHS} \
    --trainer.accelerator gpu \
    --trainer.devices 1 \
    --trainer.precision 32 \
    --trainer.default_root_dir piper_training_multi_logs \
    --data.phoneme_type text \
    --data.phonemes_path piper_training/phoneme_id_map.json \
    --data.num_symbols {num_symbols} \
    --model.num_speakers 2 \
    --model.vocoder_warmstart_ckpt "{pretrained_ckpt}"



## Part 2c — Find Best Multi-speaker Checkpoint


In [ ]:

import glob
import re
from pathlib import Path

# Multi-speaker checkpoints are stored in a dedicated log directory
multi_ckpts = sorted(
    glob.glob("piper_training_multi_logs/lightning_logs/version_*/checkpoints/*.ckpt")
)

print("Multi-speaker checkpoints found:")
for ckpt in multi_ckpts[-10:]:
    print(f"  {ckpt}")

if multi_ckpts:
    multi_best_ckpt = None
    best_val_loss   = float("inf")
    for ckpt in multi_ckpts:
        m = re.search(r"val_loss=([\d.]+)", ckpt)
        if m:
            val_loss = float(m.group(1))
            if val_loss < best_val_loss:
                best_val_loss   = val_loss
                multi_best_ckpt = ckpt

    if multi_best_ckpt:
        print(f"\nBest checkpoint (val_loss={best_val_loss:.4f}):\n  {multi_best_ckpt}")
    else:
        multi_best_ckpt = multi_ckpts[-1]
        print(f"\nNo val_loss in filenames — using latest:\n  {multi_best_ckpt}")
else:
    print("No checkpoints found. Run the training cell above first.")
    multi_best_ckpt = None



## Part 2d — Export Averaged ONNX

Loads the best multi-speaker checkpoint, averages the two speaker embeddings,
then exports as a **single-speaker** ONNX (no `sid` input) so the model is a
drop-in replacement for the regular single-speaker model.

Output: `model_piper_multi/oostfraeisk.onnx` + `.onnx.json`


In [ ]:

import torch
import json
import os
from pathlib import Path

if multi_best_ckpt is None:
    print("ERROR: No checkpoint found. Run the training and find-checkpoint cells first.")
else:
    from piper.train.vits.lightning import VitsModel

    print(f"Loading: {multi_best_ckpt}")
    model   = VitsModel.load_from_checkpoint(multi_best_ckpt, map_location="cpu")
    model_g = model.model_g
    model_g.eval()

    with torch.no_grad():
        model_g.dec.remove_weight_norm()

    assert model_g.n_speakers == 2, f"Expected n_speakers=2, got {model_g.n_speakers}"

    # Average the two learned speaker embeddings
    s0  = model_g.emb_g.weight.data[0]
    s1  = model_g.emb_g.weight.data[1]
    avg = (s0 + s1) / 2.0
    print(f"Speaker 0 embedding norm : {s0.norm().item():.4f}")
    print(f"Speaker 1 embedding norm : {s1.norm().item():.4f}")
    print(f"Averaged embedding norm  : {avg.norm().item():.4f}")

    model_g.emb_g.weight.data[0] = avg  # slot 0 now holds the average

    # Wrap infer() to hardcode sid=0 — sid is NOT exposed as an ONNX input,
    # so piper-tts will load the result as a plain single-speaker model.
    def infer_forward_averaged(text, text_lengths, scales):
        audio = model_g.infer(
            text, text_lengths,
            noise_scale=scales[0],
            length_scale=scales[1],
            noise_scale_w=scales[2],
            sid=torch.LongTensor([0]),
        )[0].unsqueeze(1)
        return audio

    model_g.forward = infer_forward_averaged  # type: ignore

    os.makedirs("model_piper_multi", exist_ok=True)
    output_path = "model_piper_multi/oostfraeisk.onnx"

    dummy_len   = 50
    sequences   = torch.randint(0, model_g.n_vocab, (1, dummy_len), dtype=torch.long)
    seq_lengths = torch.LongTensor([dummy_len])
    scales      = torch.FloatTensor([0.667, 1.0, 0.8])

    with torch.no_grad():
        torch.onnx.export(
            model=model_g,
            args=(sequences, seq_lengths, scales),
            f=output_path,
            dynamo=False,
            verbose=False,
            opset_version=15,
            input_names=["input", "input_lengths", "scales"],
            output_names=["output"],
            dynamic_axes={
                "input":         {0: "batch_size", 1: "phonemes"},
                "input_lengths": {0: "batch_size"},
                "output":        {0: "batch_size", 2: "time"},
            },
        )

    # Write config with num_speakers=1 so piper-tts treats this as single-speaker
    with open("piper_training_multi/config.json") as f:
        config = json.load(f)
    config["num_speakers"] = 1
    config.pop("speaker_id_map", None)
    with open("model_piper_multi/oostfraeisk.onnx.json", "w") as f:
        json.dump(config, f, ensure_ascii=False, indent=2)

    size_mb = os.path.getsize(output_path) / 1e6
    print(f"\nExported: {output_path}  ({size_mb:.1f} MB)")
    print("Config:   model_piper_multi/oostfraeisk.onnx.json  (num_speakers=1)")



## Part 2e — Test Multi-speaker Model

Compare the single-speaker and averaged multi-speaker outputs.


In [ ]:

from piper import PiperVoice
import wave

multi_voice = PiperVoice.load("model_piper_multi/oostfraeisk.onnx")
print(f"Multi-speaker averaged model loaded  (sample_rate={multi_voice.config.sample_rate})")

test_sentences_multi = [
    "Moin, woo gaajt 't dii?",
    "Denkent jii, dat ik disser sats gaud uutprooten dau?",
    "Hest duu däi süen fandóóeğ al säin?",
    "Däi oorsprungelk tóól fan däi Fräisen tüsken Laauwers un Wäiser was dat Olfräisk.",
]

for i, text in enumerate(test_sentences_multi):
    out_path = f"test_piper_multi_{i}.wav"
    with wave.open(out_path, "w") as wav_file:
        if hasattr(multi_voice, "synthesize_wav"):
            multi_voice.synthesize_wav(text, wav_file)
        else:
            multi_voice.synthesize(text, wav_file)
    print(f"[{i}] {text}")


In [ ]:

import IPython

print("── Single-speaker ──────────────────────────────")
display(IPython.display.Audio("test_piper_0.wav"))
print("── Multi-speaker (averaged) ────────────────────")
display(IPython.display.Audio("test_piper_multi_0.wav"))


# 10. Push to Git

In [ ]:
# Restore original metadata.csv if we modified it (espeak mode)
import shutil
from pathlib import Path

backup_path = Path("data/oostfraeisk/metadata.csv.original")
if backup_path.exists():
    shutil.copy(backup_path, "data/oostfraeisk/metadata.csv")
    print("Restored original metadata.csv")

In [ ]:
# Clean up training artifacts
!rm -rf piper_training/
!rm -rf lightning_logs/
print("Cleaned training artifacts")

In [ ]:
!git config --global user.email "programmingstudios227@gmail.com"
!git config --global user.name "Tido Specht"
!git add model_piper/
!git add -A
!git commit -m "Piper model treenäärt"
!git push

# 11. Deploy to HuggingFace Space

The HF Space needs:
- `model.onnx` + `model.onnx.json` — the exported model
- `app.py` — use `app_piper.py` from this repo
- `requirements.txt`: `piper-tts`, `gradio`

**Important:** If you trained in **grapheme mode**, set `PHONEME_MODE = "grapheme"` in `app_piper.py` — the East Frisian→German preprocessing will be skipped since the model understands native letters.

In [ ]:
!pip install huggingface_hub ipywidgets

In [ ]:
from huggingface_hub import notebook_login
notebook_login()

In [ ]:
%cd ..
!git clone https://huggingface.co/spaces/VanModers114/East_Frisian_TTS

In [ ]:
# Copy model + app to HF Space
!cp oostfraeisk_text_to_speech/model_piper/oostfraeisk.onnx East_Frisian_TTS/model.onnx
!cp oostfraeisk_text_to_speech/model_piper/oostfraeisk.onnx.json East_Frisian_TTS/model.onnx.json
!cp oostfraeisk_text_to_speech/app_piper.py East_Frisian_TTS/app.py

# Create requirements.txt
!echo "piper-tts" > East_Frisian_TTS/requirements.txt
!echo "gradio" >> East_Frisian_TTS/requirements.txt

In [ ]:
%cd East_Frisian_TTS
!git add -A
!git commit -m "Piper ONNX model"
!git push